# Optimización por Descenso del Gradiente

## Librerías y tipado

In [ ]:
# Módulos
import numpy as np
from numpy.typing import NDArray  # Para "type hints"
from typing import Callable
import matplotlib.pyplot as plt
from matplotlib import animation  # Animaciones auxiliares

# Tipos
type array = NDArray[np.float64]

## Función de Rosenbrock

La [función generalizada de Rosenbrock](https://docs.scipy.org/doc/scipy-0.14.0/reference/tutorial/optimize.html#unconstrained-minimization-of-multivariate-scalar-functions-minimize) se expresa como $
f(\mathbf{x}) = \sum _{i=1}^{N-1}\left( 100\left( x_{i} -x_{i-1}^{2}\right)^{2} +( 1-x_{i-1})^{2}\right)
$.

Obtenemos una expresión para el i-ésimo componente del gradiente de Rosenbrock $\nabla f(\mathbf{x})$ por casos:

$$\begin{aligned}
\frac{\partial f(\mathbf{x})}{\partial x_{0}} & =\frac{\partial }{\partial x_{0}}\left( 100\left( x_{1} -x_{0}^{2}\right)^{2} +( 1-x_{0})^{2} +\cdots \right)\\
 & =-400x_{0}\left( x_{1} -x_{0}^{2}\right) -2( 1-x_{0})
\end{aligned} \tag{1}$$

$$\begin{aligned}
\frac{\partial f(\mathbf{x})}{\partial x_{N-1}} & =\frac{\partial }{\partial x_{N-1}}\left( \cdots +100\left( x_{N-1} -x_{N-2}^{2}\right)^{2} +( 1-x_{N-2})^{2}\right)\\
 & =200\left( x_{N-1} -x_{N-2}^{2}\right)
\end{aligned} \tag{2}$$

$$\begin{aligned}
\forall k\in \{1,2,\dotsc ,N-2\} ,\frac{\partial f(\mathbf{x})}{\partial x_{k}} & =\frac{\partial }{\partial x_{k}}\sum _{i=1}^{N-1}\left( 100\left( x_{i} -x_{i-1}^{2}\right)^{2} +( 1-x_{i-1})^{2}\right)\\
 & \begin{align*}
=\frac{\partial }{\partial x_{k}}\Bigl[ \cdots + & \left( 100\left( x_{k} -x_{k-1}^{2}\right)^{2} +( 1-x_{k-1})^{2}\right) +\\
 & \left( 100\left( x_{k+1} -x_{k}^{2}\right)^{2} +( 1-x_{k})^{2}\right) +\cdots \Bigr]
\end{align*}\\
 & =200\left( x_{k} -x_{k-1}^{2}\right) -400x_{k}\left( x_{k+1} -x_{k}^{2}\right) -2( 1-x_{k})\\
 & =400x_{k}^{3} +( 202-400x_{k+1}) x_{k} -200x_{k-1}^{2} -2
\end{aligned} \tag{3}$$

Así, para el caso de 2 variables, utilizamos $\begin{cases}
N = 2 \\
x_0 = x \\
x_1 = y
\end{cases}$, donde no aplicaría $(3)$, y para 3 variables, $\begin{cases}
N = 3 \\
x_0 = x \\
x_1 = y \\
x_2 = z
\end{cases}$.

### Definiciones

In [ ]:
def rosenbrock(x: array):
    """
    Calcula la sumatoria de rosenbrock por medio de un generador sobre `x`.
    
    `x` puede tener `N` dimensiones."""
    N = len(x)
    return sum(
        100 * (x[i] - x[i-1]**2)**2 + (1 - x[i-1])**2
        for i in range(1, N)
    )

def rosenbrock_gradient(x: array):
    N = len(x)
    return np.array([
        -400*x[0]*(x[1] - x[0]**2) - 2*(1 - x[0]) if k == 0 else
        200 * (x[N-1] - x[N-2]**2) if k == N-1 else
        400*x[k] + (202 - 400*x[k+1])*x[k] - 200*x[k-1]**2 - 2
        for k in range(N)
    ])


LIMITS = (-5.0, 5.0)
"""
Los límites `a` y `b` para la región [a, b]² donde se genera la posición
inicial."""

def get_initial_position(size: int):
    """Genera un arreglo de números aleatorios.
    
    Los valores del arreglo tienen distribución uniforme con limites dados por
    `LIMITS`.
    """
    return np.random.uniform(*LIMITS, size)

def descend(
        position: array,
        gradient_function: Callable[[array], array],
        rate: float
    ):
    """Genera una mejor posición descendiendo desde la posición actual.
    
    Para determinar la nueva posición, se evalúa `gradient_function` en
    `position`, y se toma un paso en la dirección opuesta del gradiente
    obtenido, con su magnitud escalada por `rate`.

    Se retorna `(p_n, g, c)`, donde `p_n` es la nueva posición, `g` es el
    gradiente calculado y `c` es el paso tomado.
    """
    gradient = gradient_function(position)
    change = -rate * gradient
    return position + change, gradient, change

### 2D

#### Optimización

In [ ]:
rate = 0.0001
max_iterations = 1000
initial_position = get_initial_position(2)
position = initial_position
print('Posición Inicial:', position)
print('Valor Función Objetivo:', rosenbrock(position))
for i in range(max_iterations):
    position, gradient, change = descend(position, rosenbrock_gradient, rate)
    print(f'Iteración {i}')
    print(f'\t{gradient=}')
    print(f'\t{change=}')
    print(f'\t{position=}')

print('Posición Final:', position)
print('Valor Función Objetivo:', rosenbrock(position))

#### Animación Auxiliar

In [ ]:
samples = 100
X, Y = np.meshgrid(*[np.linspace(*LIMITS, samples)]*2)
fig, ax = plt.subplots()
ax.contour(X, Y, 100*(Y - X**2)**2 + (1 - X)**2, 100)
point = ax.plot(*initial_position, 'ro')[0]

def update(_):
    """Actualiza el punto graficado usando el gradiente descendiente.
    
    Obtiene el punto actual en el gráfico creado y aplica una iteración
    del algoritmo para retornar el punto del siguiente frame.
    """
    point.set_data(*descend(
        np.array(point.get_data()),
        rosenbrock_gradient,
        rate
    )[0])
    return [point]

ANIMATION_DURATION = 5  # in seconds
anim = animation.FuncAnimation(
    fig,
    func=update,
    frames=max_iterations,
    interval=1000 * ANIMATION_DURATION / max_iterations
)

anim.save('rosen_grad_2d.mp4')

### 3D

In [ ]:
position = get_initial_position(3)
print('Posición inicial:', position)
print('Valor Función Objetivo', rosenbrock(position))

rate_3d = 0.000_1
max_iterations_3d = 10_000
for i in range(max_iterations_3d):
    position, gradient, change = descend(position, rosenbrock_gradient, rate_3d)
    print('Iteración', i)
    print(f'\t{gradient=}')
    print(f'\t{change=}')
    print(f'\t{position=}')

print('Posición final:', position)
print('Valor Función Objetivo:', rosenbrock(position))